In [13]:
import pandas as pd
import json
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Set, Tuple
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

# ==================== Data Structures ====================

@dataclass
class Paper:
    """Represents a research paper with its metadata"""
    idx: int
    title: str
    author: str
    year: int
    core_themes: List[str]
    models_methods: List[str]
    research_scope: List[str]
    applications: List[str]

@dataclass
class ThemeNode:
    """Represents a node in the theme hierarchy"""
    name: str
    description: str
    parent: 'ThemeNode' = None
    children: List['ThemeNode'] = None
    papers: Set[int] = None
    embedding: np.ndarray = None
    
    def __post_init__(self):
        if self.children is None:
            self.children = []
        if self.papers is None:
            self.papers = set()

# ==================== Core Theme Extraction ====================

CORE_THEMES_MAPPING = {
    "人流預測": {
        "description": "Predicting and analyzing pedestrian/crowd flow patterns",
        "aliases": ["人流預測", "人流量預測", "流量預測", "crowd flow prediction", "flow forecasting", "人群流動"],
        "parent": None
    },
    "人流異常檢測": {
        "description": "Detecting and analyzing anomalies in crowd movement",
        "aliases": ["人流異常", "異常檢測", "異常預測", "anomaly detection", "outlier detection", "找出異常", "異常分析", "exception detection"],
        "parent": "人流預測"
    },
    "時空預測": {
        "description": "Spatio-temporal predictions for crowd dynamics",
        "aliases": ["時空預測", "時空分析", "空間預測", "temporal prediction", "spatio-temporal", "人群分布", "distribution prediction"],
        "parent": "人流預測"
    },
    "傳染病防治": {
        "description": "Disease prevention and epidemic management",
        "aliases": ["傳染病防治", "傳染預測", "感染控制", "disease prevention", "epidemic management", "院內感染", "infection control"],
        "parent": None
    },
    "公共管理應用": {
        "description": "Public management and emergency response",
        "aliases": ["公共管理", "交通管理", "災害管理", "公共安全", "public management", "emergency response", "救災", "disaster management"],
        "parent": None
    },
    "商業決策": {
        "description": "Business intelligence and commercial applications",
        "aliases": ["商業決策", "商業分析", "商業應用", "行銷策略", "business intelligence", "business strategy", "商圈分析"],
        "parent": "公共管理應用"
    },
    "轉移學習": {
        "description": "Transfer learning for improved model performance",
        "aliases": ["轉移學習", "遷移學習", "transfer learning", "domain adaptation", "預訓練", "fine-tuning"],
        "parent": None
    },
    "深度學習模型": {
        "description": "Deep learning architectures and methods",
        "aliases": ["深度學習", "神經網路", "深層神經網路", "deep learning", "neural networks", "machine learning", "LSTM", "CNN", "GAN"],
        "parent": None
    },
    "數據增強": {
        "description": "Data augmentation and synthetic data generation",
        "aliases": ["數據增強", "資料擴增", "合成資料", "data augmentation", "synthetic data generation", "生成資料"],
        "parent": None
    },
    "集成學習": {
        "description": "Ensemble learning methods",
        "aliases": ["集成學習", "集成方法", "ensemble learning", "ensemble methods", "multiple models"],
        "parent": "深度學習模型"
    },
    "特徵工程": {
        "description": "Feature engineering and spatial analysis",
        "aliases": ["特徵工程", "特徵提取", "feature engineering", "spatial analysis", "聚類分析", "clustering"],
        "parent": None
    }
}

class ThemeHierarchy:
    """Manages the hierarchical classification of themes with semantic embeddings"""
    
    def __init__(self, model_name: str = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
        """Initialize hierarchy with semantic model"""
        print("Loading semantic embedding model...")
        self.model = SentenceTransformer(model_name)
        self.themes: Dict[str, ThemeNode] = {}
        self._build_hierarchy()
    
    def _build_hierarchy(self):
        """Build the theme hierarchy tree with embeddings"""
        # First pass: create all nodes
        for theme_name in CORE_THEMES_MAPPING:
            node = ThemeNode(
                name=theme_name,
                description=CORE_THEMES_MAPPING[theme_name]["description"]
            )
            self.themes[theme_name] = node
        
        # Generate embeddings for each theme based on aliases
        print("Generating semantic embeddings for themes...")
        for theme_name, theme_info in CORE_THEMES_MAPPING.items():
            # Combine aliases and description for richer embedding
            text = " ".join(theme_info["aliases"]) + " " + theme_info["description"]
            embedding = self.model.encode(text, convert_to_numpy=True)
            self.themes[theme_name].embedding = embedding
        
        # Second pass: establish parent-child relationships
        for theme_name, theme_info in CORE_THEMES_MAPPING.items():
            parent_name = theme_info["parent"]
            if parent_name and parent_name in self.themes:
                parent_node = self.themes[parent_name]
                child_node = self.themes[theme_name]
                child_node.parent = parent_node
                parent_node.children.append(child_node)
    
    def get_hierarchy_structure(self) -> Dict:
        """Return the hierarchy as a nested dictionary"""
        def build_tree(node: ThemeNode) -> Dict:
            return {
                "name": node.name,
                "description": node.description,
                "paper_count": len(node.papers),
                "children": [build_tree(child) for child in node.children]
            }
        
        roots = [node for node in self.themes.values() if node.parent is None]
        return {
            "hierarchy": [build_tree(root) for root in roots],
            "total_themes": len(self.themes)
        }
    
    def add_papers_to_theme(self, theme_name: str, paper_idx: int):
        """Add a paper to a theme"""
        if theme_name in self.themes:
            self.themes[theme_name].papers.add(paper_idx)

# ==================== Paper Processing ====================

def parse_papers_from_df(df: pd.DataFrame) -> List[Paper]:
    """Convert pandas DataFrame to Paper objects"""
    papers = []
    
    for idx, row in df.iterrows():
        try:
            year = int(row['年份']) if pd.notna(row['年份']) else 0
            
            # Split multi-value fields by Chinese delimiter
            def split_field(field):
                if pd.isna(field):
                    return []
                return [t.strip() for t in str(field).split('、') if t.strip()]
            
            paper = Paper(
                idx=idx,
                title=row['題目'].strip() if pd.notna(row['題目']) else "",
                author=row['作者'].strip() if pd.notna(row['作者']) else "",
                year=year,
                core_themes=split_field(row['核心主題與目標']),
                models_methods=split_field(row['模型與方法']),
                research_scope=split_field(row['研究範圍與資料']),
                applications=split_field(row['應用'])
            )
            papers.append(paper)
        except Exception as e:
            print(f"Warning: Skipping row {idx} due to error: {e}")
            continue
    
    return papers

# ==================== Similarity Engine ====================

class SemanticSearchEngine:
    """Semantic similarity-based search engine using embeddings"""
    
    def __init__(self, papers: List[Paper], hierarchy: ThemeHierarchy):
        self.papers = papers
        self.hierarchy = hierarchy
        self.papers_dict = {p.idx: p for p in papers}
        self.model = hierarchy.model
        self._assign_papers_to_themes()
    
    def _assign_papers_to_themes(self):
        """Assign papers to themes using semantic similarity"""
        print("Assigning papers to themes using semantic similarity...")
        for paper in self.papers:
            for theme_keyword in paper.core_themes:
                # Encode the theme keyword
                kw_embedding = self.model.encode(theme_keyword, convert_to_numpy=True)
                
                # Find most similar theme
                best_theme = None
                best_sim = 0.3  # threshold
                
                for theme_name, theme_node in self.hierarchy.themes.items():
                    # Calculate cosine similarity
                    similarity = np.dot(kw_embedding, theme_node.embedding) / (
                        np.linalg.norm(kw_embedding) * np.linalg.norm(theme_node.embedding) + 1e-10
                    )
                    
                    if similarity > best_sim:
                        best_sim = similarity
                        best_theme = theme_name
                
                if best_theme:
                    self.hierarchy.add_papers_to_theme(best_theme, paper.idx)
    
    def find_best_matching_theme(self, query: str, top_k: int = 3) -> List[Tuple[str, float]]:
        """Find the best matching themes for a query using semantic similarity"""
        # Encode query
        query_embedding = self.model.encode(query, convert_to_numpy=True)
        
        # Calculate similarity with all themes
        scores = []
        for theme_name, theme_node in self.hierarchy.themes.items():
            similarity = np.dot(query_embedding, theme_node.embedding) / (
                np.linalg.norm(query_embedding) * np.linalg.norm(theme_node.embedding) + 1e-10
            )
            scores.append((theme_name, float(similarity)))
        
        # Sort by similarity and return top-k
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores[:top_k]
    
    def search_by_query(self, query: str, top_k_themes: int = 1, top_k_papers: int = 5) -> Dict:
        """Search papers based on semantic similarity to query"""
        # Find matching themes
        theme_matches = self.find_best_matching_theme(query, top_k=top_k_themes)
        
        if not theme_matches or theme_matches[0][1] < 0.3:
            return {
                "status": "no_match",
                "message": f"No semantically similar theme found for: '{query}'",
                "query": query,
                "papers": []
            }
        
        # Collect papers from matched themes and their descendants
        all_paper_indices = set()
        matched_themes_info = []
        
        for theme_name, similarity in theme_matches:
            theme_node = self.hierarchy.themes[theme_name]
            matched_themes_info.append({
                "theme": theme_name,
                "similarity": round(similarity, 3)
            })
            
            # Get papers from this theme and descendants
            def get_descendant_papers(node: ThemeNode) -> Set[int]:
                all_papers = set(node.papers)
                for child in node.children:
                    all_papers.update(get_descendant_papers(child))
                return all_papers
            
            all_paper_indices.update(get_descendant_papers(theme_node))
        
        # Sort papers by year (newest first)
        results = sorted(
            [self.papers_dict[i] for i in all_paper_indices],
            key=lambda p: p.year,
            reverse=True
        )[:top_k_papers]
        
        return {
            "status": "success",
            "query": query,
            "matched_themes": matched_themes_info,
            "paper_count": len(results),
            "papers": [
                {
                    "title": p.title,
                    "author": p.author,
                    "year": p.year,
                    "core_themes": p.core_themes,
                    "applications": p.applications
                }
                for p in results
            ]
        }
    
    def search_by_semantic_keyword(self, keyword: str, top_k: int = 10) -> Dict:
        """Search papers using semantic similarity to keyword"""
        keyword_embedding = self.model.encode(keyword, convert_to_numpy=True)
        matching_papers = []
        
        for paper in self.papers:
            score = 0
            
            # Title similarity (highest weight)
            if paper.title:
                title_emb = self.model.encode(paper.title, convert_to_numpy=True)
                title_sim = np.dot(keyword_embedding, title_emb) / (
                    np.linalg.norm(keyword_embedding) * np.linalg.norm(title_emb) + 1e-10
                )
                score += title_sim * 3
            
            # Methods similarity
            if paper.models_methods:
                methods_text = " ".join(paper.models_methods)
                methods_emb = self.model.encode(methods_text, convert_to_numpy=True)
                methods_sim = np.dot(keyword_embedding, methods_emb) / (
                    np.linalg.norm(keyword_embedding) * np.linalg.norm(methods_emb) + 1e-10
                )
                score += methods_sim * 2
            
            # Applications similarity
            if paper.applications:
                apps_text = " ".join(paper.applications)
                apps_emb = self.model.encode(apps_text, convert_to_numpy=True)
                apps_sim = np.dot(keyword_embedding, apps_emb) / (
                    np.linalg.norm(keyword_embedding) * np.linalg.norm(apps_emb) + 1e-10
                )
                score += apps_sim * 2
            
            # Core themes similarity
            if paper.core_themes:
                themes_text = " ".join(paper.core_themes)
                themes_emb = self.model.encode(themes_text, convert_to_numpy=True)
                themes_sim = np.dot(keyword_embedding, themes_emb) / (
                    np.linalg.norm(keyword_embedding) * np.linalg.norm(themes_emb) + 1e-10
                )
                score += themes_sim
            
            if score > 0:
                matching_papers.append((paper, score))
        
        matching_papers.sort(key=lambda x: x[1], reverse=True)
        
        return {
            "keyword": keyword,
            "results_count": len(matching_papers),
            "papers": [
                {
                    "title": p.title,
                    "author": p.author,
                    "year": p.year,
                    "relevance_score": round(score, 3)
                }
                for p, score in matching_papers[:top_k]
            ]
        }

# ==================== Main System ====================

def main(csv_path: str = './paper_entries.csv'):
    """Main function to run the search system"""
    
    # Load CSV
    print(f"\n{'='*80}")
    print(f"Loading papers from {csv_path}...")
    print(f"{'='*80}\n")
    try:
        df = pd.read_csv(csv_path, encoding='utf-8')
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return
    
    # Parse papers
    papers = parse_papers_from_df(df)
    if not papers:
        print("No papers loaded. Please check the CSV file.")
        return
    
    print(f"✓ Loaded {len(papers)} papers\n")
    
    # Initialize hierarchy with semantic model
    hierarchy = ThemeHierarchy()
    print("✓ Theme hierarchy built with semantic embeddings\n")
    
    # Create search engine
    search_engine = SemanticSearchEngine(papers, hierarchy)
    print("✓ Search engine initialized\n")
    
    # # ==================== Demo ====================
    print("=" * 80)
    print("THEME HIERARCHY STRUCTURE")
    print("=" * 80)
    hierarchy_dict = hierarchy.get_hierarchy_structure()
    print(json.dumps(hierarchy_dict, ensure_ascii=False, indent=2))
    
    print("\n" + "=" * 80)
    # print("SEMANTIC SIMILARITY SEARCH - DEMONSTRATION")
    # print("=" * 80)
    
    # Test with various rephrasing examples
    test_queries = [
        "找出人群流動中的異常狀況",  # Rephrasing of 異常檢測
        "outlier detection in crowd",  # English version
        "未來人群分布的模擬",  # Rephrasing of 時空預測
        "disease spread prevention",  # Rephrase of 傳染病防治
        "深層神經網路建模",  # Rephrase of 深度學習
        "商業智能決策系統",  # Rephrase of 商業決策
        "ensemble model methods",  # English for 集成學習
    ]
    
    for query in test_queries:
        print(f"\n>>> Query: '{query}'")
        results = search_engine.search_by_query(query, top_k_themes=2, top_k_papers=2)
        
        if results['status'] == 'success':
            print(f"    Matched Themes:")
            for theme_info in results['matched_themes']:
                print(f"      • {theme_info['theme']} (similarity: {theme_info['similarity']})")
            print(f"    Papers Found: {results['paper_count']}")
            if results['papers']:
                for i, p in enumerate(results['papers'], 1):
                    print(f"      {i}. {p['title'][:50]}... ({p['author']}, {p['year']})")
        else:
            print(f"    {results['message']}")
    
    print("\n" + "=" * 80)
    print("SEMANTIC KEYWORD SEARCH - DEMONSTRATION")
    print("=" * 80)
    
    keywords = ["LSTM", "異常檢測", "epidemic", "人流"]
    for keyword in keywords:
        print(f"\n>>> Keyword: '{keyword}'")
        kw_results = search_engine.search_by_semantic_keyword(keyword, top_k=3)
        print(f"    Results Found: {kw_results['results_count']}")
        if kw_results['papers']:
            for i, p in enumerate(kw_results['papers'], 1):
                print(f"      {i}. {p['title'][:50]}... ({p['author']}) - Score: {p['relevance_score']}")
    
    print("\n" + "=" * 80)
    print("SYSTEM SUMMARY")
    print("=" * 80)
    print(f"Total papers: {len(papers)}")
    print(f"Total themes: {len(hierarchy.themes)}")
    print("\nPapers by theme:")
    for theme_name in sorted(hierarchy.themes.keys()):
        theme_node = hierarchy.themes[theme_name]
        if theme_node.papers:
            print(f"  • {theme_name}: {len(theme_node.papers)} papers")

if __name__ == "__main__":
    main()


Loading papers from ./paper_entries.csv...

✓ Loaded 6 papers

Loading semantic embedding model...
Generating semantic embeddings for themes...
✓ Theme hierarchy built with semantic embeddings

Assigning papers to themes using semantic similarity...
✓ Search engine initialized

THEME HIERARCHY STRUCTURE
{
  "hierarchy": [
    {
      "name": "人流預測",
      "description": "Predicting and analyzing pedestrian/crowd flow patterns",
      "paper_count": 5,
      "children": [
        {
          "name": "人流異常檢測",
          "description": "Detecting and analyzing anomalies in crowd movement",
          "paper_count": 0,
          "children": []
        },
        {
          "name": "時空預測",
          "description": "Spatio-temporal predictions for crowd dynamics",
          "paper_count": 1,
          "children": []
        }
      ]
    },
    {
      "name": "傳染病防治",
      "description": "Disease prevention and epidemic management",
      "paper_count": 1,
      "children": []
    },
    

In [6]:
import pandas as pd
pd.read_csv('./paper_entries.csv')

,題目,作者,年份,核心主題與目標,模型與方法,研究範圍與資料,應用
0,應用補償式遷移學習模型於區域人流之強健性預測,李香蘭,2022,區域人流預測、都市人流量、強健性預測、補償式遷移學習模型,遷移學習、集成學習、CNN-LSTM、卷積神經網路、長短期記憶模型、RBF-LSTM、徑向基...,雙北市之商業區、行動數據人流資料、Google搜尋趨勢(Google Trends),商業決策、公共衛生管理、交通管制、救災
1,基於R-tree與SPACE-MDL-LSTM提升大區域人流預測之效率,鄭力誠,2021,人流預測、提升大區域人流預測之效率、改善過往小區域建模成本過高之缺點、解決大區域單一模型預測...,R-tree、SPACE-MDL-LSTM、LSTM(長短期記憶模型)、MDL(最小描述長度...,北北基桃宜地區、行動數據人流資料,商業決策、交通管理、公共安全、傳染病防治、降低建模成本
2,基於集成式生成對抗網路進行人流異常預測,王詠緹,2021,人流異常預測分析、提升人流異常的預測準確性、處理異常資料不足的問題、解決依賴標記數據的問題,集成式學習 (Ensemble Learning)、生成對抗網路 (GAN)、半監督式學習、...,北台灣部分商圈、行動數據人流資料,交通管理、災害管理、資源分配、公共安全、商業銷售策略
3,基於人流資料、土地使用分區圖以及Google趨勢資料進行捷運進出站人數預測-以台北捷運為例,盧政傑,2021,捷運進出站人數預測、解決新設站點無歷史資料無法預測的問題、降低訓練時間成本/運算成本、捷運站...,地理空間特徵分類、集成式學習 (Ensemble Learning)、HRNN (Hamme...,台北捷運每日進出站人數資料、行動數據人流資料、都市計畫使用分區資料、Google搜尋趨勢(G...,捷運乘客流量預測、商業行銷策略、災害緊急應變措施、捷運班次調度與人潮疏散
4,基於3D-RCL與條件式生成對抗網路產生未來時刻人群分布之可能性探討,鄭佳昇,2022,人群分布預測與模擬、解決歷史資料過少的問題、生成未來時刻人群分布、模擬區域內不同影響因子,條件式生成對抗網路 (cGAN)、資料擴增 (Data Augmentation)、特徵工程...,臺北市及新北市、行動數據人流資料、天氣資料、日曆資料,災防管理、人群兵棋推演
5,"利用RBF-DNN配合醫院病患人流資料探究流行病之院內感染熱區""",施長宏,2023,流行病之院內感染熱區、傳染預測、院內感染、降低院內發生感染的機率,RBF-DNN、RBF函數、深度學習、Arena模擬、合成資料、隨機森林,台大斗六院區病患靠卡時間資料、臺大醫院雲林分院一樓平面圖,準確匡列可能的接觸者、院內感染熱區、制定感染預防政策、分流看診
